# Tutorial 6: Train NicheTrans* on STARmap PLUS data

In [ ]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_ct import *
from datasets.data_manager_STARmap_PLUS import AD_Mouse

from utils.utils import *
from utils.notebook_hparams import build_model_from_args
from utils.utils_training_STARmap_PLUS import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [ ]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

### Initialize dataloaders and NicheTrans

In [ ]:
# create the dataloaders
dataset = AD_Mouse(AD_adata_path=args.AD_adata_path, Wild_type_adata_path=args.Wild_type_adata_path, n_top_genes=args.n_top_genes)
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.target_length
model = build_model_from_args(NicheTrans_ct, args, source_length=source_dimension, target_length=target_dimension)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

### Initialize loss function (criterion) and optimizer

In [ ]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Optional MoE Trajectory Tracking During Training

Enable the next cell if you want to monitor whether expert usage becomes more balanced or more specialized across epochs.


In [ ]:
from utils.moe_analysis import (
    analyze_moe_routing,
    summarize_epoch_trajectory,
    summarize_layerwise_epoch_trajectory,
)

track_moe_during_training = False
moe_track_every = 1
moe_track_max_batches = None  # Set to a small integer for faster monitoring on large datasets.
moe_epoch_frames = {}
moe_epoch_layer_frames = {}
moe_epoch_overall = []


### Model training and testing

In [ ]:
start_time = time.time()

if "track_moe_during_training" not in globals():
    track_moe_during_training = False
    moe_track_every = 1
    moe_track_max_batches = None
    moe_epoch_frames = {}
    moe_epoch_layer_frames = {}
    moe_epoch_overall = []

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, ct_information=True, device=device)
    if args.stepsize > 0: scheduler.step()

    if track_moe_during_training and ((epoch + 1) % moe_track_every == 0 or last_epoch):
        moe_epoch_result = analyze_moe_routing(
            model=model,
            dataloader=testloader,
            device=device,
            include_images=False,
            include_cell_information=True,
            include_predictions=False,
            include_targets=False,
            max_batches=moe_track_max_batches,
            add_spatial_regions=False,
        )
        moe_epoch_frames[epoch + 1] = moe_epoch_result["activation_frame"]
        moe_epoch_layer_frames[epoch + 1] = moe_epoch_result["layer_activation_frames"]
        moe_epoch_overall.append({"epoch": epoch + 1, **moe_epoch_result["overall"]})
        print("MoE tracking at epoch {}:".format(epoch + 1), moe_epoch_result["overall"])
    ################

pearson = test(model, testloader, ct_information=True, device=device)
torch.save(model.state_dict(), 'NicheTrans_*_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))


### Optional MoE Routing Analysis

Run the next cell after training if you want to inspect expert activation, load balance, and spatial specialization.


In [ ]:
from utils.moe_analysis import (
    analyze_moe_routing,
    plot_all_experts_spatial_grid,
    plot_center_spot_activation_bar,
    plot_expert_spatial_heatmap,
    plot_slice_activation_heatmap,
    save_moe_analysis_tables,
)

moe_results = analyze_moe_routing(
    model=model,
    dataloader=testloader,
    device=device,
    include_images=False,
    include_cell_information=True,
    include_predictions=False,
    include_targets=True,
)

activation_frame = moe_results["activation_frame"]  # Final layer for backward compatibility.
layer_results = moe_results["layer_results"]
layer_activation_frames = moe_results["layer_activation_frames"]
layer_overall_summary = moe_results["layer_overall_summary"]

print("Overall MoE metrics (final layer for backward compatibility):")
print(moe_results["overall"])
if not layer_overall_summary.empty:
    print("Layer-wise MoE metrics:")
    display(layer_overall_summary)

available_layers = sorted(
    layer_activation_frames,
    key=lambda name: int(str(name).split("_")[-1]) if str(name).split("_")[-1].isdigit() else str(name),
)

for layer_name in available_layers:
    print()
    print(f"=== {layer_name} ===")
    layer_metrics = layer_results.get(layer_name, {})
    layer_frame = layer_metrics.get("activation_frame", layer_activation_frames[layer_name])

    if layer_metrics.get("overall"):
        print("Layer overall:")
        print(layer_metrics["overall"])
    if "expert_summary" in layer_metrics and not layer_metrics["expert_summary"].empty:
        print("Expert summary:")
        display(layer_metrics["expert_summary"])
    if "slice_summary" in layer_metrics and not layer_metrics["slice_summary"].empty:
        print("Slice summary:")
        display(layer_metrics["slice_summary"])
    if "region_summary" in layer_metrics and not layer_metrics["region_summary"].empty:
        print("Region summary:")
        display(layer_metrics["region_summary"])

    display(layer_frame.head())
    plot_center_spot_activation_bar(layer_frame, row_index=0);

    if layer_frame["x"].notna().any() and layer_frame["y"].notna().any():
        first_slice = layer_frame["slice_id"].dropna().iloc[0]
        plot_slice_activation_heatmap(layer_frame, slice_id=first_slice);
        plot_all_experts_spatial_grid(layer_frame, slice_id=first_slice, columns=4);
    else:
        print(
            f"Spatial coordinates were not recovered for {layer_name}. "
            "Pass `sample_metadata_resolver` to `analyze_moe_routing(...)` if you want spatial heatmaps."
        )

# Optional: save the analysis tables to disk (includes layer-wise files for multi-layer MoE).
# save_moe_analysis_tables(moe_results, output_dir="./moe_analysis")


### Optional MoE Training Trajectory Summary

If epoch-level tracking was enabled during training, the next cell summarizes how expert usage changed over time.


In [ ]:
import math
import matplotlib.pyplot as plt
from utils.moe_analysis import summarize_epoch_trajectory, summarize_layerwise_epoch_trajectory

trajectory_metric_specs = [
    ("usage_entropy_normalised", "Usage Entropy", "Normalized entropy"),
    ("top1_entropy_normalised", "Top1 Entropy", "Normalized entropy"),
    ("effective_expert_count", "Effective Expert Count", "Effective experts"),
    ("dominant_expert_fraction", "Dominant Expert Fraction", "Dominant mass"),
    ("mean_gate_margin", "Mean Gate Margin", "Margin"),
    ("std_gate_margin", "Gate Margin Std", "Std"),
    ("mean_spot_entropy", "Mean Spot Entropy", "Entropy"),
    ("std_spot_entropy", "Spot Entropy Std", "Std"),
    ("balance_loss", "Balance Loss", "Loss"),
    ("router_entropy_penalty", "Router Entropy Penalty", "Penalty"),
    ("expert_output_cosine_mean", "Expert Output Cosine Mean", "Cosine"),
    ("expert_output_cosine_std", "Expert Output Cosine Std", "Std"),
]


def plot_metric_grid(frame, metric_specs, title_prefix=""):
    if frame.empty:
        print(f"No metrics available for {title_prefix.strip() or 'trajectory'} plots.")
        return

    available_specs = [spec for spec in metric_specs if spec[0] in frame.columns]
    if not available_specs:
        print(f"No matching metrics found for {title_prefix.strip() or 'trajectory'} plots.")
        return

    ncols = 3
    nrows = math.ceil(len(available_specs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows), squeeze=False)

    for axis, (metric, title, ylabel) in zip(axes.flat, available_specs):
        axis.plot(frame["epoch"], frame[metric], marker="o")
        axis.set_title(f"{title_prefix}{title}")
        axis.set_xlabel("Epoch")
        axis.set_ylabel(ylabel)

    for axis in axes.flat[len(available_specs):]:
        axis.axis("off")

    plt.tight_layout()


def plot_layer_compare_grid(by_layer, metric_specs):
    if not by_layer:
        print("No layer-wise trajectory is available for comparison plots.")
        return

    available_specs = [
        spec for spec in metric_specs
        if any(spec[0] in layer_frame.columns for layer_frame in by_layer.values())
    ]
    if not available_specs:
        print("No matching metrics found for layer-wise comparison plots.")
        return

    ncols = 3
    nrows = math.ceil(len(available_specs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows), squeeze=False)

    for axis, (metric, title, ylabel) in zip(axes.flat, available_specs):
        for layer_name, layer_frame in by_layer.items():
            if metric in layer_frame.columns:
                axis.plot(layer_frame["epoch"], layer_frame[metric], marker="o", label=layer_name)
        axis.set_title(f"Layer-wise {title}")
        axis.set_xlabel("Epoch")
        axis.set_ylabel(ylabel)
        axis.legend()

    for axis in axes.flat[len(available_specs):]:
        axis.axis("off")

    plt.tight_layout()


moe_epoch_layer_frames = globals().get("moe_epoch_layer_frames", {})

if not moe_epoch_frames:
    print(
        "No epoch-level MoE trajectory was recorded. "
        "Set `track_moe_during_training = True` before training and rerun the notebook if you want this summary."
    )
else:
    final_layer_epoch_trajectory = summarize_epoch_trajectory(moe_epoch_frames)
    layerwise_moe_epoch_trajectory = summarize_layerwise_epoch_trajectory(moe_epoch_layer_frames)
    overall_epoch_trajectory = layerwise_moe_epoch_trajectory["epoch_summary"]
    if overall_epoch_trajectory.empty:
        overall_epoch_trajectory = final_layer_epoch_trajectory

    print("Overall MoE trajectory:")
    display(overall_epoch_trajectory)

    if not final_layer_epoch_trajectory.empty:
        print("Final-layer trajectory (backward compatibility):")
        display(final_layer_epoch_trajectory)

    if not layerwise_moe_epoch_trajectory["combined"].empty:
        print("Layer-wise MoE trajectory:")
        display(layerwise_moe_epoch_trajectory["combined"])

    for layer_name, layer_frame in layerwise_moe_epoch_trajectory["by_layer"].items():
        print(f"{layer_name} trajectory:")
        display(layer_frame)

    plot_metric_grid(overall_epoch_trajectory, trajectory_metric_specs, title_prefix="Overall ")
    plot_layer_compare_grid(layerwise_moe_epoch_trajectory["by_layer"], trajectory_metric_specs)

# Optional: save the trajectory tables to disk.
# overall_epoch_trajectory.to_csv("./moe_analysis/moe_epoch_trajectory.csv", index=False)
# layerwise_moe_epoch_trajectory["combined"].to_csv("./moe_analysis/moe_epoch_layer_trajectory.csv", index=False)
